# 03_rag_pipeline

Steps: <br>
1. User question 
2. Embedding search (S3 Vectors) 
3. Get recipe_id(s) + s3_key 
4. Fetch JSON from S3 
5. Build CONTEXT STRING 
6. Send to LLM (Bedrock)
7. Answer

In [1]:
import json
import boto3
from IPython.display import display, Markdown
import base64

In [2]:
s3 = boto3.client("s3")
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
vector_client = boto3.client("s3vectors", region_name="us-east-1")


# RAG Functions

In [31]:
def embed_text(text):
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "inputText": text,
            "dimensions": 1024,
            "normalize": True
        })
    )

    result = json.loads(response["body"].read())
    return result["embedding"]

In [32]:
def build_image_prompt(recipe):
    return f"""
A professionally photographed dish of {recipe['recipe_title']}.
The dish is a modern gourmet interpretation of a historic American recipe.
Plated beautifully on a rustic wooden table with warm natural lighting.
Ingredients include: {', '.join(recipe.get('recipe_ingredients', [])[:6])}.
Warm natural lighting, rustic table setting, high detail, food photography.
"""

In [33]:
def generate_recipe_image(prompt):
    # Titan uses 'taskType' and 'textToImageParams'
    body = json.dumps({
        "taskType": "TEXT_IMAGE",
        "textToImageParams": {
            "text": prompt
        },
        "imageGenerationConfig": {
            "numberOfImages": 1,
            "height": 1024,
            "width": 1024,
            "cfgScale": 8.0,
            "seed": 0
        }
    })

    response = bedrock.invoke_model(
        modelId="amazon.titan-image-generator-v2:0", 
        body=body,
        accept="application/json",
        contentType="application/json"
    )

    response_body = json.loads(response.get("body").read().decode("utf-8"))
    
    # Titan returns a list called 'images' containing base64 strings
    base64_image = response_body.get("images")[0]
    
    return base64_image

In [34]:
def run_rag(query, n_recipes=3):

    # Embed query
    query_embedding = embed_text(query)

    # vector search 
    response = vector_client.query_vectors(
        vectorBucketName="recipe-vector-bucket",
        indexName="recipe-index",
        queryVector={"float32": query_embedding},
        topK=10,
        returnMetadata=True
    )

    # Deduplicate recipes
    recipes = {}
    seen = set()

    for v in response["vectors"]:
        meta = v.get("metadata", {})

        recipe_id = meta.get("recipe_id")
        s3_key = meta.get("s3_key")

        if not recipe_id or recipe_id in seen:
            continue

        seen.add(recipe_id)

        # fetch full recipe from S3
        obj = s3.get_object(
            Bucket="feeding-america-historic-cookbooks",
            Key=s3_key
        )

        book = json.loads(obj["Body"].read())
        recipe = book["recipes"].get(recipe_id)

        if not recipe:
            continue

        recipes[recipe_id] = {
            "recipe": recipe,
            "book_metadata": book.get("metadata", {})
        }

        if len(recipes) == n_recipes:
            break
            
    # Build LLM context
    context_parts = []

    for recipe_id, data in recipes.items():
        r = data["recipe"]
        book_meta = data.get("book_metadata", {})

        context_parts.append(f"""
            Title: {r.get('recipe_title')}
            Book: {book_meta.get('book_title', 'Unknown')}
            Author: {book_meta.get('book_creator', 'Unknown')}
            Year: {book_meta.get('book_year', 'Unknown')}
            
            Ingredients:
            {', '.join(r.get('recipe_ingredients', []))}
            
            Original Instructions:
            {r.get('recipe_instructions')}
            
            --------------------
            """)

    context = "\n".join(context_parts)

    # ------------------------
    # 5. LLM prompt (Nova / Bedrock)
    # ------------------------
    prompt = f"""
        You are a cooking assistant specializing in historic American recipes from the late 18th to early 20th century.
        
        Use ONLY the provided context. Do NOT invent metadata.
        
        Context:
        {context}
        
        User Question:
        {query}
        
        Instructions:
        - Return top {n_recipes} recipes with no repeats
        - Use ONLY provided metadata
        - If missing, write "Unknown"
        
        Format each recipe as:
        
        Recipe 1:
        Title:
        Book:
        Author:
        Year Published:
        
        Headnote:
        ...
        
        Ingredients:
        - ...
        
        Original Instructions:
        ...
        
        Modern Modified Instructions:
        1. ...
        """

    llm_response = bedrock.invoke_model(
        modelId="amazon.nova-micro-v1:0",
        contentType="application/json",
        body=json.dumps({
            "messages": [
                {
                    "role": "user",
                    "content": [{"text": prompt}]
                }
            ]
        })
    )

    result = json.loads(llm_response["body"].read())
    output = result["output"]["message"]["content"][0]["text"]

    return output

    # image generation. FIND A MODEL THAT WORKS
    '''images = []
    image_meta = []

    for recipe_id, data in recipes.items():

        recipe = data["recipe"]

        image_prompt = build_image_prompt(recipe)

        image_b64 = generate_recipe_image(image_prompt)

         
        #save images to s3 bucket?
        image = base64.b64decode(image_b64) 

        images.append(image)

        image_meta.append({
            "recipe_id": recipe_id,
            "title": recipe.get("recipe_title")
        })


    return final_text, images, image_meta'''

# Test RAG Pipeline

In [35]:
result = run_rag("Vegetable soups")
print(type(result))
print(len(result))

<class 'str'>
5290


In [36]:
print(run_rag("Vegetable soups"))

Here are the top three recipes for vegetable soups from the provided context, formatted as requested:

### Recipe 1:
**Title:** VEGETABLE SOUP (FROM STOCK)  
**Book:** Manual For Army Cooks...  
**Author:** United States War Department.  
**Year Published:** 1896.

**Headnote:**  
A traditional soup made from stock, featuring a mix of vegetables that can be adjusted based on availability.

**Ingredients:**  
- stock  
- salt  
- pepper  
- boiling water  
- rice  

**Original Instructions:**  
VEGETABLE SOUP (FROM STOCK). 1 gallon stock. 4 lbs. mixed vegetables (about). Salt and pepper. Prepare the vegetables as directed in Remarks on Soup, put them into a pot of boiling water slightly salted, and just enough to cover them, and boil until cooked. About ten minutes before the vegetables are cooked, put on the stock and bring it to a boil, then stir in the cooked vegetables, and, in order that they may not stick to the bottom, keep stirring the soup until it boils up; season lightly and 